In [1]:
"""
Method 6: Iterative Pruning — ResNet-50 / CIFAR-10
===================================================
Iterative pruning progressively removes weights in multiple rounds,
evaluating accuracy at each step. Contrasts with one-shot pruning
which removes all weights in a single pass.

Schedule: each round removes PRUNING_STEP fraction of REMAINING non-zero
weights (global L1 magnitude), until MAX_SPARSITY is reached.

Saves exactly 8 metrics per round for comparison with baseline:
  accuracy, precision, recall, f1, num_parameters, model_size_mb,
  inference_time_ms, flops

Output: method6_iterative_pruning.json
"""

import os, json, time, copy, tempfile, warnings
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from thop import profile

warnings.filterwarnings("ignore")

# ── Config ────────────────────────────────────────────────────────────────────
BASELINE_MODEL_PATH   = "../__2__baseline_resnet50_cifar10.pth"
BASELINE_METRICS_PATH = "../__2__baseline_metrics.json"
OUTPUT_JSON           = "method6_iterative_pruning.json"

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE  = 128
NUM_WORKERS = 2
NUM_CLASSES = 10

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

PRUNING_STEP   = 0.15   # remove 15% of REMAINING non-zero weights per round
MAX_SPARSITY   = 0.90   # stop when overall sparsity reaches this level
INFERENCE_RUNS = 100


# ── Model ─────────────────────────────────────────────────────────────────────
def build_model(num_classes=10):
    model = models.resnet50(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model

def load_baseline(path):
    model = build_model(NUM_CLASSES).to(DEVICE)
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.eval()
    return model


# ── Data ──────────────────────────────────────────────────────────────────────
def get_test_loader():
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(root="../data", train=False,
                                       download=True, transform=transform)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)


# ── Pruning ───────────────────────────────────────────────────────────────────
def prune_one_step(model, step_amount):
    """
    Remove step_amount fraction of CURRENT non-zero weights globally (L1).
    Operates in-place on the model.
    """
    all_weights = torch.cat([
        m.weight.data[m.weight.data != 0].abs().view(-1)
        for _, m in model.named_modules()
        if isinstance(m, (nn.Conv2d, nn.Linear))
    ])
    if all_weights.numel() == 0:
        return model

    k         = max(1, int(all_weights.numel() * step_amount))
    threshold = torch.kthvalue(all_weights, k).values.item()

    with torch.no_grad():
        for _, module in model.named_modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)):
                mask = (module.weight.data.abs() > threshold).float()
                module.weight.data *= mask
    return model

def real_sparsity(model):
    zeros = total = 0
    for _, m in model.named_modules():
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            zeros += (m.weight == 0).sum().item()
            total += m.weight.numel()
    return zeros / total if total else 0.0


# ── 8 Core Metrics ────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    for inputs, lbls in loader:
        inputs = inputs.to(DEVICE, non_blocking=True)
        preds.extend(model(inputs).argmax(1).cpu().numpy())
        labels.extend(lbls.numpy())
    y_pred, y_true = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

def get_model_size_mb(model):
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(model.state_dict(), tmp)
        size = os.path.getsize(tmp) / (1024 ** 2)
    finally:
        os.remove(tmp)
    return size

def measure_inference_time_ms(model):
    model.eval()
    dummy = torch.randn(BATCH_SIZE, 3, 32, 32).to(DEVICE)

    if DEVICE.type == "cuda":
        for _ in range(20):
            model(dummy)
        torch.cuda.synchronize()
        start = torch.cuda.Event(enable_timing=True)
        end   = torch.cuda.Event(enable_timing=True)
        start.record()
        with torch.no_grad():
            for _ in range(INFERENCE_RUNS):
                model(dummy)
        end.record()
        torch.cuda.synchronize()
        return start.elapsed_time(end) / INFERENCE_RUNS
    else:
        with torch.no_grad():
            for _ in range(5):
                model(dummy)
            t0 = time.perf_counter()
            for _ in range(INFERENCE_RUNS):
                model(dummy)
        return (time.perf_counter() - t0) / INFERENCE_RUNS * 1000

def compute_flops(model):
    model.eval()
    dummy = torch.randn(1, 3, 32, 32).to(DEVICE)
    macs, _ = profile(model, inputs=(dummy,), verbose=False)
    return int(macs * 2)


# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    print(f"\n{'='*62}")
    print("  Method 6: Iterative Pruning — ResNet-50 / CIFAR-10")
    print(f"  Device : {DEVICE}")
    print(f"  Step: {int(PRUNING_STEP*100)}% of remaining per round | "
          f"Max sparsity: {int(MAX_SPARSITY*100)}%")
    print(f"{'='*62}\n")

    with open(BASELINE_METRICS_PATH) as f:
        baseline = json.load(f)

    model  = load_baseline(BASELINE_MODEL_PATH)
    loader = get_test_loader()

    current   = copy.deepcopy(model)
    results   = []
    round_num = 0

    while True:
        round_num += 1
        print(f"  Round {round_num}: pruning {int(PRUNING_STEP*100)}% of remaining weights...")
        current = prune_one_step(current, PRUNING_STEP)
        sp      = real_sparsity(current)

        metrics       = evaluate(current, loader)
        num_params    = count_parameters(current)
        model_size_mb = get_model_size_mb(current)
        inference_ms  = measure_inference_time_ms(current)
        flops         = compute_flops(current)

        row = {
            "round"             : round_num,
            "cumulative_sparsity": round(sp, 4),
            # ── 8 standardized metrics ──
            "accuracy"         : round(metrics["accuracy"],  6),
            "precision"        : round(metrics["precision"], 6),
            "recall"           : round(metrics["recall"],    6),
            "f1"               : round(metrics["f1"],        6),
            "num_parameters"   : num_params,
            "model_size_mb"    : round(model_size_mb, 4),
            "inference_time_ms": round(inference_ms, 4),
            "flops"            : flops,
        }
        results.append(row)

        print(f"    Sparsity: {sp*100:.1f}% | Acc: {metrics['accuracy']:.4f} | "
              f"Size: {model_size_mb:.2f} MB | Inf: {inference_ms:.2f} ms | "
              f"FLOPs: {flops/1e9:.3f} G")

        if sp >= MAX_SPARSITY:
            print(f"\n  Reached max sparsity ({MAX_SPARSITY*100:.0f}%). Stopping.")
            break
        if round_num >= 30:
            print("\n  Safety cap reached. Stopping.")
            break

    report = {
        "method"     : "iterative_pruning",
        "description": (f"Iterative L1 magnitude pruning: {int(PRUNING_STEP*100)}% "
                        "of remaining non-zero weights removed per round (global L1). "
                        "Each result row = one pruning round with cumulative sparsity."),
        "pruning_step": PRUNING_STEP,
        "total_rounds": round_num,
        "baseline"    : baseline,
        "results"     : results,
    }

    with open(OUTPUT_JSON, "w") as f:
        json.dump(report, f, indent=2)
    print(f"\n  ✓ Saved → {OUTPUT_JSON}")
    print(f"  Total rounds: {round_num} | "
          f"Final sparsity: {results[-1]['cumulative_sparsity']*100:.1f}%")


if __name__ == "__main__":
    main()


  Method 6: Iterative Pruning — ResNet-50 / CIFAR-10
  Device : cuda
  Step: 15% of remaining per round | Max sparsity: 90%

  Round 1: pruning 15% of remaining weights...
    Sparsity: 15.0% | Acc: 0.9321 | Size: 90.03 MB | Inf: 103.75 ms | FLOPs: 2.623 G
  Round 2: pruning 15% of remaining weights...
    Sparsity: 27.7% | Acc: 0.9321 | Size: 90.04 MB | Inf: 63.23 ms | FLOPs: 2.623 G
  Round 3: pruning 15% of remaining weights...
    Sparsity: 38.6% | Acc: 0.9320 | Size: 90.04 MB | Inf: 352.55 ms | FLOPs: 2.623 G
  Round 4: pruning 15% of remaining weights...
    Sparsity: 47.8% | Acc: 0.9321 | Size: 90.04 MB | Inf: 344.86 ms | FLOPs: 2.623 G
  Round 5: pruning 15% of remaining weights...
    Sparsity: 55.6% | Acc: 0.9320 | Size: 90.04 MB | Inf: 381.04 ms | FLOPs: 2.623 G
  Round 6: pruning 15% of remaining weights...
    Sparsity: 62.3% | Acc: 0.9322 | Size: 90.04 MB | Inf: 343.98 ms | FLOPs: 2.623 G
  Round 7: pruning 15% of remaining weights...
    Sparsity: 67.9% | Acc: 0.9320 | 